# download

> Download `facebook/wav2vec2-base-960h` and write a numpy checkpoint.
> 
> The Hugging Face repo ships `model.safetensors`, so conversion does not
> need PyTorch. It folds the weight-norm pair into one weight and writes the
> file the runtime loads, next to the character vocabulary.


In [ ]:
#| default_exp download


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from pathlib import Path

import numpy as np

from nada.core import repo_root
from nada.weights import load_arrays


In [ ]:
#| export
REPO = "facebook/wav2vec2-base-960h"


In [ ]:
#| export
def download(root: Path | None = None) -> Path:
    """Fetch the config and vocabulary, convert the weights, and return the safetensors path."""
    from huggingface_hub import snapshot_download

    destination = (root or repo_root()) / "models" / "wav2vec2-base-960h"
    destination.mkdir(parents=True, exist_ok=True)
    snapshot_download(REPO, local_dir=destination, allow_patterns=["*.json", "model.safetensors"])
    output = (root or repo_root()) / "models" / "wav2vec2.safetensors"
    convert(destination / "model.safetensors", output)
    return output


In [ ]:
#| export
def convert(source: Path, output: Path) -> None:
    """Drop buffers the graph does not store, fold weight norm, and write float32 safetensors."""
    from safetensors.numpy import save_file

    arrays = {key: np_contiguous(value) for key, value in load_arrays(source).items()}
    output.parent.mkdir(parents=True, exist_ok=True)
    save_file(arrays, output)
    print(f"wrote {output} ({len(arrays)} tensors)")


In [ ]:
#| export
def np_contiguous(array):
    if np.issubdtype(array.dtype, np.floating):
        array = array.astype(np.float32, copy=False)
    return np.ascontiguousarray(array)


In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    download()
